# Usage

A quick guide to `block_formats` usage.

**Imports**

By convention, we use `import block_formats.quantisation as Q`, etc.

In [1]:
%env WANDB_SILENT=true
%env TOKENIZERS_PARALLELISM=true
%load_ext autoreload
%autoreload 2

from pathlib import Path
import torch

import block_formats.analysis as A
import block_formats.experiments as E
import block_formats.fit as F
import block_formats.quantisation as Q
import block_formats.sensitivity as S

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env: WANDB_SILENT=true
env: TOKENIZERS_PARALLELISM=true


**Load a model and dataset (with calculation of reference output)**

In [2]:
model = E.RequantisableModel.load("meta-llama/Llama-3.2-1B", DEVICE, torch.bfloat16)
params = {k: v.detach() for k, v in model.model.named_parameters() if v.ndim == 2}

# For a quick test - 1 batch of shape (16, 256)
data = E.Dataset.load_wikitext2(model.model, sequence_length=256, batch_size=16, kl_topk=128, sequence_limit=16)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


**Quantise a single parameter, and evaluate**

In [3]:
model.reset()

p = model.model.state_dict()['model.layers.15.self_attn.v_proj.weight']
fmt = Q.CompressedLUTFormat.train_grid(p, p.std().item() / 4)
print(fmt.count_bits_tensor(p) / p.nelement(), "bpp")
print("R =", Q.qrmse_norm(fmt, p).cpu())
p[...] = fmt.quantise(p)

display(data.evaluate(model.model))

4.0 bpp
R = tensor(0.0724)


{'cross_entropy': tensor([2.5975, 2.6224, 3.1826, 2.2801, 3.3747, 2.7373, 3.1326, 2.9880, 3.0249,
         2.5361, 3.1598, 3.0744, 2.7587, 2.9885, 3.0168, 2.8383],
        device='cuda:0'),
 'kl_div': tensor([0.0010, 0.0010, 0.0009, 0.0008, 0.0010, 0.0009, 0.0010, 0.0010, 0.0010,
         0.0009, 0.0009, 0.0010, 0.0009, 0.0010, 0.0009, 0.0010],
        device='cuda:0')}

**Quantise all parameters using `F.Scaled.fit`**

This takes a few seconds, as it runs k-means per weight tensor.

In [5]:
log = model.quantise({"": F.Scaled(4, "lloyd_max", Q.BFLOAT16, (1, 64), "absmax", compressor=None, args=dict(init="kmeans++"))})
display(log[0])
print(log[0]["bits"] / log[0]["nelement"], "bpp")

{'name': 'model.embed_tokens.weight',
 'quantised': True,
 'nelement': 262668288,
 'bits': 1116340224,
 'rmse': 0.0020667058415710926,
 'norm': 0.021906794980168343,
 'fmt': {'element_format': {'values': (-0.9663415460053096,
    -0.7780264147879061,
    -0.6300980183854988,
    -0.5038973805524972,
    -0.39158404823052234,
    -0.28825852121329176,
    -0.19062699857561216,
    -0.09656785694771618,
    -0.004124155621101294,
    0.0885144992256883,
    0.18463751386739058,
    0.2878806801189355,
    0.4030956374258181,
    0.5373025235838832,
    0.7039978191995361,
    0.9363573461998741),
   'name': 'LM',
   '_type': 'lut'},
  'scale_format': {'dtype': 'bfloat16', '_type': 'torch'},
  'block_shape': (1, 64),
  'scaling': 'absmax',
  '_type': 'linear'},
 'fmt_str': 'LUT4[LM]{1,64:BFLOAT16:absmax}'}

4.25 bpp


**Run a tiny sweep**

In [4]:
!rm out/dev.jsonl

E.TokenPredictionSweep(
    experiment="dev",
    formats=[{}, {"": F.Scaled(4, "fp", Q.BFLOAT16, (1, 32), "rms", compressor=None)}],
    models=E.TEST_MODELS[:1],
).run(Path("out/dev.jsonl"))

-- model 1/1, format 1/2


-- model 1/1, format 2/2
